In [1]:
# -*- coding: utf-8 -*-
import sys, os
print("Python:", sys.version.split()[0])

Python: 3.12.4


In [2]:
# 加载大模型
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os
load_dotenv()

if os.getenv("ARK_API_KEY"):
    ark_model = os.getenv("ARK_MODEL", "doubao-seed-evolving") #采用doubao-seed-evolving性能最适合智能体
    #ark_model = os.getenv("ARK_MODEL_2", "doubao-seed-2-1-turbo-260628") #采用doubao-seed-2-1-turbo-260628，速度快
    #ark_model = os.getenv("ARK_MODEL_3", "doubao-seed-2-1-pro-260628")  #采用doubao-seed-2-1-pro-260628
    #ark_model = os.getenv("ARK_MODEL_4", "deepseek-v4-pro-ga-260813")  #采用deepseek-v4-pro-ga-260813，速度慢
    ark_base = os.getenv("ARK_BASE_URL", "https://ark.cn-beijing.volces.com/api/v3")
    llm = ChatOpenAI(
        model=ark_model,
        api_key=os.getenv("ARK_API_KEY"),
        base_url=ark_base,
        temperature=0,
    )
    print(f"使用豆包大模型 (火山引擎) | base_url={ark_base} | model={ark_model}")
else:
    llm = ChatOpenAI(
        model=os.getenv("DEEPSEEK_MODEL", "deepseek-v4-pro"),
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url=os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"),
    )
    print("使用 DeepSeek 大模型")



# 连通性自检：提前暴露订阅/密钥/模型名问题，避免在 agent 调用时才报错
resp = llm.invoke("请只回复两个字：正常")
print("[OK] LLM 连通正常:", resp.content[:50])

使用豆包大模型 (火山引擎) | base_url=https://ark.cn-beijing.volces.com/api/v3 | model=doubao-seed-evolving
[OK] LLM 连通正常: 正常


In [3]:
# 创建 Deep Agent，加载两个技能包
# deepagents 0.7.x 的 LocalShellBackend 没有 python_path 参数，需用 inherit_env=True 继承当前进程环境(PATH)，否则子进程 cmd 找不到 python
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langgraph.checkpoint.memory import InMemorySaver

_py_dir = os.path.dirname(sys.executable)

agent = create_deep_agent(
    model=llm,
    backend=LocalShellBackend(
        root_dir=".",
        virtual_mode=True,
        inherit_env=True,  # 继承 os.environ（含 PATH），修复"找不到 Python 解释器"
        env={"PATH": _py_dir + os.pathsep + os.path.join(_py_dir, "Scripts") + os.pathsep + os.environ.get("PATH", "")},
    ),
    skills=["./skills"],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "你是结构力学机动分析助手。对结构示意图的分析按以下顺序输出：\n"
        "第1步 先读图并绘制标注结构示意图：看清节点、杆件、支座，按 doubao-visualization 规范输出一张 HTML/SVG 示意图，"
        "      标出节点编号、所有杆件、支座符号（固定铰=2 根链杆、可动铰=1 根）及约束说明。\n"
        "      图是可直接渲染的 ```html 代码块。图的背景取白色。"
        "第2步 再做机动分析：确认节点数 j、杆件数 b、支座链杆数 r -> 计算自由度 W（桁架 W=2j-b-r；刚架 W=3m-2h-r），"
        "      用 freedom_degree 脚本复核 -> 用二元体/两刚片/三刚片法则做几何组成分析（注意三铰共线、无穷远铰等退化情形）\n"
        "      -> 结论：几何不变(静定/超静定)/瞬变/常变 + 多余约束数。\n"
        
    ),
)
print("Deep Agent 创建完成")

Deep Agent 创建完成


In [4]:
import time
t0 = time.time()

from langchain_core.messages import HumanMessage

IMAGE = "./truss-1.png"   # 结构示意图路径
#IMAGE = "./truss-2.png"   # 结构示意图路径
question = (
    f"对结构示意图 {IMAGE} 做结构力学机动分析。\n\n"
    "第1步：先仔细看清图片，按 doubao-visualization 规范绘制一张标注结构示意图（HTML/SVG renderer，```html 代码块）——"
    "标出全部节点编号、全部杆件、支座符号（固定铰/可动铰及各自链杆数）。图中所有文字均采用英文。\n"
    "第2步：再做机动分析——确认 j/b/r -> 计算 W（用 freedom_degree 脚本复核）-> "
    "用二元体/两刚片/三刚片法则做几何组成分析 -> 结论（几何不变/瞬变/常变 + 多余约束数 + 静定/超静定）。"
)

result = agent.invoke(
    {"messages": [HumanMessage(content=question)]},
    config={"configurable": {"thread_id": "analysis-1"}},
)

for m in result["messages"]:
    if m.type == "ai" and m.content:
        print(m.content)
        print("\n" + "-" * 60)

print(f"agent 耗时: {(time.time()-t0)/60:.1f}m")

我先读取判定规则、例题库，并运行图像解析脚本交叉核对节点与杆件。


------------------------------------------------------------
我已看清图片。现在运行图像解析脚本与自由度脚本交叉核对，同时读取 HTML/SVG 绘图规范。


------------------------------------------------------------
让我定位图片文件实际路径并运行解析脚本。


------------------------------------------------------------
Python 可用。先运行自由度脚本复核，再运行图像解析脚本（显示完整报错）。


------------------------------------------------------------
这是 Windows 环境，虚拟路径与真实盘符不一致。我读取脚本内容并写到可执行位置运行。


------------------------------------------------------------
工作目录在 H 盘。我把 freedom_degree 脚本写到简单路径后运行复核。


------------------------------------------------------------
直接用内联 Python 复核自由度，避免路径映射问题。


------------------------------------------------------------
W=0 已确认。现在尝试直接运行 skill 自带的 freedom_degree.py 脚本完成正式复核。


------------------------------------------------------------
脚本文件在虚拟层，物理路径不存在。我用与 freedom_degree.py 完全一致的逻辑内联运行，得到脚本的完整判定输出。


------------------------------------------------------------
已完成读图、计数与自由度复核。下面按要求分两步给出。

---

## 第 1 步　标注结构示意图

读图要

In [5]:
# 第三步：在 notebook 内直接渲染 agent 输出的标注示意图（遍历所有 AI 消息，避免取错）
def extract_renderer_html(text):
    """从 agent 回复中提取 ```html 代码块内容（用于 notebook 内渲染）"""
    import re
    m = re.search(r"```html[^\n]*\n(.*?)```", text, re.S)
    return m.group(1) if m else None



from IPython.display import display, HTML
html_code = None
for m in reversed(result["messages"]):
    if m.type == "ai" and m.content:
        html_code = extract_renderer_html(m.content)
        if html_code:
            break

if html_code:
    display(HTML(html_code))
else:
    print("未找到 HTML 代码块。消息诊断（最后 6 条）：")
    for i, m in enumerate(result["messages"][-6:]):
        c = (str(m.content)[:150] if m.content else "(空)")
        print(f"  [{i}] type={m.type} | content={c}")

In [ ]:
#交互式对话，保持 Agent 记忆，连续提问（例如："把右侧支座去掉还稳定吗？"）。
while True:
    q = input("User message (输入 exit 退出): ")
    if q.lower() == "exit":
        break
    res = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config={"configurable": {"thread_id": "chat-1"}},
    )
    for m in res["messages"]:
        if m.type == "ai" and m.content:
            print(m.content)
            print("\n" + "-" * 60)